# Semiconductor Yield Risk Prediction and Data-Driven Model Optimization

## Project Objective

반도체 제조 과정에서 수집되는 다수의 공정 측정값을 활용하여
생산 개체의 Pass/Fail을 예측하고, 데이터 전처리 및 모델링 전략에
따른 불량 검출 성능의 변화를 체계적으로 분석한다.

본 프로젝트에서는 단순한 모델 성능 비교를 넘어,

- 결측치 처리
- 저분산 Feature 제거
- Feature 간 상관관계 기반 Feature Selection
- Class Imbalance 대응을 위한 Sampling
- 모델별 최적 설정 탐색
- Classification Threshold 조정

등의 각 요소를 독립적으로 실험하고, Cross-Validation을 기반으로
후보 설정을 선정한다.

이후 선정된 설정을 조합하여 최종 모델을 구성하고,
독립적인 Test set에서 최종 성능을 평가한다.

최종적으로 불량 검출 성능과 오검출 사이의 Trade-off를 고려하여
반도체 제조 품질 및 수율 개선에 활용 가능한 데이터 기반
예측 접근법을 탐색하는 것을 목적으로 한다.

# Data Exploration

## 1. Library Import

In [1]:
import pandas as pd

from data.loader import load_dataset
from analysis.profiler import DataProfiler

## 2. Data Load

In [2]:
df = load_dataset("uci-secom.csv")
df.head()

,Time,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Pass/Fail
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


## 3. Dataset Summary

In [3]:
profiler = DataProfiler(df)
profiler.dataset_summary()

{'rows': 1567,
 'columns': 592,
 'memory (mb)': np.float64(7.18),
 'duplicate rows': 0,
 'total missing cells': 41951,
 'total missing ratio (%)': 4.52}

## 4. Feature Summary

In [38]:
features_summary = profiler.features_summary()
features_summary

,dtype,non_null_count,missing count per columns,missing ratio per columns (%),constant,unique
Time,object,1567,0,0.00,False,1534
0,float64,1561,6,0.38,False,1520
1,float64,1560,7,0.45,False,1504
2,float64,1553,14,0.89,False,507
3,float64,1553,14,0.89,False,518
...,...,...,...,...,...,...
586,float64,1566,1,0.06,False,322
587,float64,1566,1,0.06,False,260
588,float64,1566,1,0.06,False,120
589,float64,1566,1,0.06,False,611


In [41]:
features_summary[(features_summary['missing ratio per columns (%)'] > 5) & (features_summary['constant'] == 'True')]

,dtype,non_null_count,missing count per columns,missing ratio per columns (%),constant,unique


In [5]:
profiler.features_summary()['dtype'].value_counts()

dtype
float64    590
object       1
int64        1
Name: count, dtype: int64

## 5. Missing Values per Column

In [18]:
summary = profiler.missing_columns()
summary

,missing count per columns,missing ratio per columns (%)
158,1429,91.19
292,1429,91.19
293,1429,91.19
157,1429,91.19
85,1341,85.58
...,...,...
386,0,0.00
361,0,0.00
360,0,0.00
359,0,0.00


In [35]:
summary[summary['missing ratio per columns (%)'] <= 16]

,missing count per columns,missing ratio per columns (%)
90,51,3.25
89,51,3.25
363,51,3.25
362,51,3.25
225,51,3.25
...,...,...
386,0,0.00
361,0,0.00
360,0,0.00
359,0,0.00


In [36]:
summary[summary['missing ratio per columns (%)'] > 16]

,missing count per columns,missing ratio per columns (%)
158,1429,91.19
292,1429,91.19
293,1429,91.19
157,1429,91.19
85,1341,85.58
492,1341,85.58
220,1341,85.58
358,1341,85.58
517,1018,64.96
245,1018,64.96


## 6. Missing Values per Row

In [7]:
profiler.missing_rows()

,missing count per rows,missing ratio per rows (%)
1566,152,25.68
1564,148,25.00
1561,140,23.65
1152,100,16.89
511,100,16.89
...,...,...
773,4,0.68
1455,4,0.68
843,4,0.68
1208,4,0.68


## 7. Target Summary

In [8]:
profiler.target_summary('Pass/Fail')

{'dtype': dtype('int64'),
 'non_null_count': np.int64(1567),
 'missing count': 0,
 'missing ratio (%)': 0.0,
 'unique': 2,
 'value_counts': {-1: 1463, 1: 104},
 'value_counts_ratio (%)': {-1: 93.36, 1: 6.64}}

## Conclusion

### Dataset

 - 데이터셋은 1567개의 샘플(rows)과 592개의 컬럼(features)으로 구성되어 있음.

 - 날짜와 시간 정보가 담긴 'Time'과 타겟인 'Pass/Fail'을 제외한 데이터는 모두 센서의 측정값 또는 수치화된 범주형 값으로 추정됨.

 - 데이터 보안상의 이유로 각 값의 출처는 제공되지 않아 도메인 기반 해석에는 한계가 있음.

### Missing Value

 - 결측치는 일부 컬럼에 집중적으로 존재.

 - 전체 592개 컬럼 중 540개 컬럼은 결측률이 3.25%이하였으며, 52개 컬럼은 그 이상의 결측률을 보임.

 - 일부 컬럼은 80% 이상의 매우 높은 결측률을 보였으며, 해당 컬럼들은 모두 constant feature가 아님.

 - 결측률과 데이터 특성을 고려한 다양한 결측치 처리 전략의 성능을 비교할 예정.

### Target

 - 데이터 타입은 'Time'은 object, 'Pass/Fail'은 int, 나머지는 float.

 - Pass(-1)는 1463개(93.36%), Fail(1)은 104개(6.64%) 존재.

 - 클래스 불균형으로 인해 소수 클래스(Fail)의 재현율 저하가 발생할 가능성이 높음.

 - 모델링 단계에서 SMOTE, 클래스 가중치 등 다양한 불균형 처리 기법의 효과를 비교할 예정.